In [3]:
import pandas as pd
import requests
from io import StringIO
import certifi
import os

# SSL fix
os.environ['SSL_CERT_FILE'] = certifi.where()
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()

# Wikipedia's User-Agent policy requires identifying your bot.
# Best practice: include project name, version, and a contact (email or URL).
HEADERS = {
    'User-Agent': 'FootballIsLifeBot/0.1 (Personal soccer-analysis learning project; contact: cookyousfi@personal-email)'
}

def fetch_wikipedia_tables(url):
    """Fetch a Wikipedia page with a proper User-Agent and return all tables."""
    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()  # raises HTTPError if status >= 400
    return pd.read_html(StringIO(response.text))

def survey_afcon(year):
    url = f"https://en.wikipedia.org/wiki/{year}_Africa_Cup_of_Nations"
    print(f"\n{'='*70}\nAFCON {year}  ({url})\n{'='*70}")
    try:
        tables = fetch_wikipedia_tables(url)
    except Exception as e:
        print(f"  ERROR: {e}")
        return None
    print(f"  {len(tables)} tables total")
    match_keywords = ['home', 'away', 'score', 'result', 'date', 'venue', 'attendance']
    candidates = [
        (i, t, sum(1 for kw in match_keywords if kw in ' '.join(str(c).lower() for c in t.columns)))
        for i, t in enumerate(tables)
    ]
    candidates.sort(key=lambda x: -x[2])
    print(f"\n  Top match-like tables:")
    for i, t, hits in candidates[:8]:
        print(f"\n  Table {i}  ({hits} match-keywords, {t.shape[0]} rows × {t.shape[1]} cols)")
        print(f"    Columns: {list(t.columns)[:10]}")
        if len(t) > 0:
            print(f"    First row: {t.iloc[0].to_list()[:6]}")
    return tables

tables_2023 = survey_afcon(2023)
tables_2025 = survey_afcon(2025)


AFCON 2023  (https://en.wikipedia.org/wiki/2023_Africa_Cup_of_Nations)
  80 tables total

  Top match-like tables:

  Table 5  (1 match-keywords, 24 rows × 6 cols)
    Columns: ['Team', 'Method of qualification', 'Date of qualification', 'Finals appearance', 'Last appearance', 'Previous best performance']
    First row: ['Ivory Coast', 'Hosts / Group H runners-up', '30 January 2019', '25th', np.int64(2021), 'Champions (1992, 2015)']

  Table 0  (0 match-keywords, 21 rows × 2 cols)
    Columns: ["Coupe d'Afrique des Nations 2023", "Coupe d'Afrique des Nations 2023.1"]
    First row: ['Official logo[1]', 'Official logo[1]']

  Table 1  (0 match-keywords, 4 rows × 3 cols)
    Columns: ['Title sponsor', 'Official sponsors', 'National sponsors']
    First row: ['TotalEnergies 1XBET Rexona Visa Puma Orange', "Air Côte d'Ivoire[17] Apsonic[18] Ecobank[19] Razzl Tecno[20]", 'Celeste Porteo[21] LONACI[22] Smart Technologies[23]']

  Table 2  (0 match-keywords, 1 rows × 2 cols)
    Columns: [0,

In [4]:
def find_large_tables(tables, year, min_rows=30):
    print(f"\n=== AFCON {year} — tables with {min_rows}+ rows ===\n")
    found_any = False
    for i, t in enumerate(tables):
        if t.shape[0] >= min_rows:
            found_any = True
            cols = list(t.columns)
            print(f"Table {i}: {t.shape[0]} rows × {t.shape[1]} cols")
            print(f"  Columns: {cols[:10]}")
            if len(t) > 0:
                print(f"  First row: {t.iloc[0].to_list()[:8]}")
            if len(t) > 1:
                print(f"  Row #5:    {t.iloc[min(5, len(t)-1)].to_list()[:8]}")
            print()
    if not found_any:
        print(f"  No tables with {min_rows}+ rows. Maybe try min_rows=20?")

find_large_tables(tables_2023, 2023, min_rows=30)
find_large_tables(tables_2025, 2025, min_rows=30)


=== AFCON 2023 — tables with 30+ rows ===

Table 51: 59 rows × 15 cols
  Columns: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
  First row: [np.float64(nan), 'Round of 16', 'Round of 16', np.float64(nan), np.float64(nan), 'Quarter-finals', 'Quarter-finals', np.float64(nan)]
  Row #5:    [np.float64(nan), 'Nigeria', '2', np.float64(nan), np.float64(nan), '2 February – Abidjan (H.B)', '2 February – Abidjan (H.B)', np.float64(nan)]

Table 74: 45 rows × 3 cols
  Columns: ['Territory', 'Rights holder(s)', 'Ref.']
  First row: ['Andorra', 'LaLiga+', '[121]']
  Row #5:    ['Brazil', 'Band', '[125][100]']


=== AFCON 2025 — tables with 30+ rows ===

Table 4: 38 rows × 4 cols
  Columns: ['Country', 'Referee', 'Assistant referees', 'Matches assigned']
  First row: ['Algeria', 'Mustapha Ghorbal', 'Mahmoud Ahmed Abouelregal  Jerson Emiliano Dos Santos\xa0[de]', 'Ivory Coast–Cameroon (Group F)']
  Row #5:    ['Burundi', 'Pacifique Ndabihawenimana', 'Modibo Samake  Diana Chikotesha\xa0[fr; de]', 'Equatorial Guin

In [6]:
import re

SCORE_PATTERN = re.compile(r'^\d+\s*[\-–]\s*\d+(\s*\(.*\))?$')

def find_score_tables(tables, year, min_scores=3):
    print(f"\n=== AFCON {year} — tables containing actual scores ===\n")
   
    hits = []
    for i, t in enumerate(tables):
        n_scores = 0
        sample = None
        # Flatten the table into a 1D array of all cells.
        # t.values is a numpy 2D array; .flatten() makes it 1D for easy iteration.
        for val in t.values.flatten():
            # pd.isna() catches NaN, NaT, and None — universal "missing value" check.
            if pd.isna(val):
                continue
            val_str = str(val).strip()
            if SCORE_PATTERN.match(val_str):
                n_scores += 1
                if sample is None:
                    sample = val_str
        if n_scores >= min_scores:
            hits.append((i, t, n_scores, sample))
   
    hits.sort(key=lambda x: -x[2])
   
    print(f"Found {len(hits)} score-bearing tables. Top 15:")
    for i, t, n, sample in hits[:15]:
        print(f"\nTable {i}: {t.shape[0]} rows × {t.shape[1]} cols  "
              f"({n} score cells; sample={sample!r})")
        print(f"  Columns: {list(t.columns)[:8]}")
        if len(t) > 0:
            print(f"  First row: {t.iloc[0].to_list()[:8]}")

find_score_tables(tables_2023, 2023)
find_score_tables(tables_2025, 2025)


=== AFCON 2023 — tables containing actual scores ===

Found 0 score-bearing tables. Top 15:

=== AFCON 2025 — tables containing actual scores ===

Found 1 score-bearing tables. Top 15:

Table 69: 53 rows × 5 cols  (52 score cells; sample='2–0')
  Columns: [('Stage', 'Group stage matches'), ('Team 1', 'Group stage matches'), ('Result', 'Group stage matches'), ('Team 2', 'Group stage matches'), ('Man of the Match', 'Group stage matches')]
  First row: ['Group A', 'Morocco', '2–0', 'Comoros', 'Brahim Díaz']


In [7]:
# Test fetching one group's sub-article. If clean, we iterate all 6 groups.
url = "https://en.wikipedia.org/wiki/2023_Africa_Cup_of_Nations_Group_A"
print(f"Fetching: {url}\n")

try:
    group_a_tables = fetch_wikipedia_tables(url)
    print(f"Returned {len(group_a_tables)} tables\n")
   
    for i, t in enumerate(group_a_tables):
        cols = list(t.columns)
        print(f"Table {i}: {t.shape[0]} rows × {t.shape[1]} cols")
        print(f"  Columns: {cols[:8]}")
        if len(t) > 0:
            print(f"  First row: {t.iloc[0].to_list()[:8]}")
        if len(t) > 1:
            print(f"  Last row:  {t.iloc[-1].to_list()[:8]}")
        print()
except Exception as e:
    print(f"ERROR: {e}")

Fetching: https://en.wikipedia.org/wiki/2023_Africa_Cup_of_Nations_Group_A

Returned 40 tables

Table 0: 4 rows × 10 cols
  Columns: [('Draw position', 'Draw position'), ('Team', 'Team'), ('Zone', 'Zone'), ('Method of qualification', 'Method of qualification'), ('Date of qualification', 'Date of qualification'), ('Finals appearance', 'Finals appearance'), ('Last appearance', 'Last appearance'), ('Previous best performance', 'Previous best performance')]
  First row: ['A1', 'Ivory Coast', 'WAFU', 'Hosts and Group H runners-up', '30 January 2019', '25th', np.int64(2021), 'Winners (1992, 2015)']
  Last row:  ['A4', 'Guinea-Bissau', 'WAFU', 'Group A runners-up', '18 June 2023', '4th', np.int64(2021), 'Group stage (2017, 2019, 2021)']

Table 1: 4 rows × 11 cols
  Columns: ['Pos', 'Teamvte', 'Pld', 'W', 'D', 'L', 'GF', 'GA']
  First row: [np.int64(1), 'Equatorial Guinea', np.int64(3), np.int64(2), np.int64(1), np.int64(0), np.int64(9), np.int64(3)]
  Last row:  [np.int64(4), 'Guinea-Bissau',

In [8]:
import re

# Capture groups so we can extract home_score and away_score directly
SCORE_PATTERN = re.compile(r'^(\d+)\s*[\-–]\s*(\d+)(?:\s*\(.*\))?$')

def parse_match_tables(tables, stage_label=None):
    """
    Extract matches from Wikipedia group/knockout pages.
    Each match is a (1, 3) table where columns = [home_team, 'X-Y' score, away_team].
    """
    matches = []
    for t in tables:
        if t.shape != (1, 3):
            continue  # Wrong shape — not a match table
       
        cols = list(t.columns)
        # Some Wikipedia tables use MultiIndex columns (tuples); flatten if needed
        cols = [c[0] if isinstance(c, tuple) else c for c in cols]
       
        score_cell = str(cols[1]).strip()
        m = SCORE_PATTERN.match(score_cell)
        if not m:
            continue  # Middle column isn't a score string
       
        home_team = str(cols[0]).strip()
        away_team = str(cols[2]).strip()
       
        # Skip rows where team names are missing/junk
        if home_team in ('', 'nan') or away_team in ('', 'nan'):
            continue
       
        matches.append({
            'stage': stage_label,
            'home_team': home_team,
            'away_team': away_team,
            'home_score': int(m.group(1)),
            'away_score': int(m.group(2)),
            'score_raw': score_cell,  # keep original in case of penalty shootouts
        })
    return matches


# Smoke test on Group A
group_a_matches = parse_match_tables(group_a_tables, stage_label='Group A')
print(f"Group A: parsed {len(group_a_matches)} matches\n")
for m in group_a_matches:
    print(f"  [{m['stage']:8s}] {m['home_team']:20s} {m['home_score']}–{m['away_score']} {m['away_team']}")

Group A: parsed 6 matches

  [Group A ] Ivory Coast          2–0 Guinea-Bissau
  [Group A ] Nigeria              1–1 Equatorial Guinea
  [Group A ] Equatorial Guinea    4–2 Guinea-Bissau
  [Group A ] Ivory Coast          0–1 Nigeria
  [Group A ] Equatorial Guinea    4–0 Ivory Coast
  [Group A ] Guinea-Bissau        0–1 Nigeria


In [9]:
import time
import pandas as pd

def fetch_afcon_group_stage(year, groups=('A', 'B', 'C', 'D', 'E', 'F')):
    """Pull all group-stage matches for an AFCON tournament."""
    all_matches = []
    for letter in groups:
        url = f"https://en.wikipedia.org/wiki/{year}_Africa_Cup_of_Nations_Group_{letter}"
        print(f"  Group {letter}: fetching...", end=' ', flush=True)
        try:
            tables = fetch_wikipedia_tables(url)
        except Exception as e:
            print(f"FAILED: {e}")
            continue
        matches = parse_match_tables(tables, stage_label=f'Group {letter}')
        for m in matches:
            m['year'] = year
            m['tournament'] = 'AFCON'
        print(f"{len(matches)} matches")
        all_matches.extend(matches)
        # Be polite to Wikipedia — 1 second pause between requests
        time.sleep(1)
    return all_matches


print("=== AFCON 2023 group stage ===\n")
afcon_2023_groups = fetch_afcon_group_stage(2023)
print(f"\nTotal: {len(afcon_2023_groups)} group-stage matches")

# Convert to DataFrame for easier filtering
df_2023 = pd.DataFrame(afcon_2023_groups)

# Pull Morocco's matches specifically
morocco_mask = (df_2023['home_team'] == 'Morocco') | (df_2023['away_team'] == 'Morocco')
morocco_2023 = df_2023[morocco_mask]
print(f"\nMorocco's AFCON 2023 group stage ({len(morocco_2023)} matches):")
for _, m in morocco_2023.iterrows():
    print(f"  [{m['stage']}] {m['home_team']} {m['home_score']}–{m['away_score']} {m['away_team']}")

# Show the full DataFrame head for sanity
print("\nFull DataFrame (first 10 rows):")
print(df_2023.head(10).to_string(index=False))

=== AFCON 2023 group stage ===

  Group A: fetching... 6 matches
  Group B: fetching... 6 matches
  Group C: fetching... 6 matches
  Group D: fetching... 6 matches
  Group E: fetching... 6 matches
  Group F: fetching... 6 matches

Total: 36 group-stage matches

Morocco's AFCON 2023 group stage (3 matches):
  [Group F] Morocco 3–0 Tanzania
  [Group F] Morocco 1–1 DR Congo
  [Group F] Zambia 0–1 Morocco

Full DataFrame (first 10 rows):
  stage         home_team         away_team  home_score  away_score score_raw  year tournament
Group A       Ivory Coast     Guinea-Bissau           2           0       2–0  2023      AFCON
Group A           Nigeria Equatorial Guinea           1           1       1–1  2023      AFCON
Group A Equatorial Guinea     Guinea-Bissau           4           2       4–2  2023      AFCON
Group A       Ivory Coast           Nigeria           0           1       0–1  2023      AFCON
Group A Equatorial Guinea       Ivory Coast           4           0       4–0  2023    

In [10]:
print("=== AFCON 2023 knockout stage ===\n")
ko_url = "https://en.wikipedia.org/wiki/2023_Africa_Cup_of_Nations_knockout_stage"
print(f"Fetching: {ko_url}\n")

try:
    ko_tables = fetch_wikipedia_tables(ko_url)
    print(f"Returned {len(ko_tables)} tables\n")
except Exception as e:
    print(f"ERROR: {e}")
    ko_tables = []

# Same parser, no stage label yet — we'll assign by position in the bracket
ko_matches = parse_match_tables(ko_tables, stage_label=None)
print(f"Matches parsed: {len(ko_matches)}\n")

# AFCON knockouts are: 8 R16 + 4 QF + 2 SF + 1 3rd-place + 1 Final = 16 matches
# If we got exactly 16, the parser nailed it. If more/fewer, we may need filtering.
for i, m in enumerate(ko_matches):
    print(f"  {i+1:2d}. {m['home_team']} {m['home_score']}–{m['away_score']} {m['away_team']}  (raw={m['score_raw']!r})")

=== AFCON 2023 knockout stage ===

Fetching: https://en.wikipedia.org/wiki/2023_Africa_Cup_of_Nations_knockout_stage

Returned 100 tables

Matches parsed: 11

   1. Angola 3–0 Namibia  (raw='3–0')
   2. Nigeria 2–0 Cameroon  (raw='2–0')
   3. Equatorial Guinea 0–1 Guinea  (raw='0–1')
   4. Cape Verde 1–0 Mauritania  (raw='1–0')
   5. Mali 2–1 Burkina Faso  (raw='2–1')
   6. Morocco 0–2 South Africa  (raw='0–2')
   7. Nigeria 1–0 Angola  (raw='1–0')
   8. DR Congo 3–1 Guinea  (raw='3–1')
   9. Mali 1–2 Ivory Coast  (raw='1–2 (a.e.t.)')
  10. Ivory Coast 1–0 DR Congo  (raw='1–0')
  11. Nigeria 1–2 Ivory Coast  (raw='1–2')


In [13]:
def parse_consolidated_match_table(t, focus_team='Morocco', tournament='AFCON', year=None):
    """
    Parse Wikipedia's consolidated match table format (like AFCON 2025 Table 69),
    where matches are rows and the score is a cell value (not a column header).
    Filters to only matches involving the focus team.
    """
    # Flatten MultiIndex columns if present
    flat_cols = [c[0] if isinstance(c, tuple) else c for c in t.columns]
    t = t.copy()
    t.columns = flat_cols
   
    matches = []
    for _, row in t.iterrows():
        team1 = str(row.get('Team 1', '')).strip()
        team2 = str(row.get('Team 2', '')).strip()
       
        # Filter: only matches with our focus team in them
        if focus_team not in (team1, team2):
            continue
       
        result = str(row.get('Result', '')).strip()
        m = SCORE_PATTERN.match(result)
        if not m:
            continue
       
        # Strip "(p)" penalty winner suffixes from team names
        team1_clean = re.sub(r'\s*\(p\)\s*$', '', team1).strip()
        team2_clean = re.sub(r'\s*\(p\)\s*$', '', team2).strip()
       
        matches.append({
            'tournament': tournament,
            'year': year,
            'stage': str(row.get('Stage', '')).strip(),
            'home_team': team1_clean,
            'away_team': team2_clean,
            'home_score': int(m.group(1)),
            'away_score': int(m.group(3)),
            'home_pen': int(m.group(2)) if m.group(2) else None,
            'away_pen': int(m.group(4)) if m.group(4) else None,
            'penalty_shootout': (m.group(2) is not None and m.group(4) is not None),
            'motm': str(row.get('Man of the Match', '')).strip(),
        })
    return matches


# Use the AFCON 2025 tables we already fetched earlier in the notebook
table_69 = tables_2025[69]
morocco_2025 = parse_consolidated_match_table(
    table_69, focus_team='Morocco', tournament='AFCON', year=2025
)

print(f"Morocco's AFCON 2025 matches ({len(morocco_2025)}):\n")
for m in morocco_2025:
    pen_note = f"  [P {m['home_pen']}-{m['away_pen']}]" if m['penalty_shootout'] else ""
    print(f"  [{m['stage']:18s}] {m['home_team']:20s} {m['home_score']}–{m['away_score']} {m['away_team']:20s}{pen_note}")
    if m['motm']:
        print(f"    MOTM: {m['motm']}")

Morocco's AFCON 2025 matches (6):

  [Group A           ] Morocco              2–0 Comoros             
    MOTM: Brahim Díaz
  [Group A           ] Morocco              1–1 Mali                
    MOTM: Neil El Aynaoui
  [Group A           ] Zambia               0–3 Morocco             
    MOTM: Ayoub El Kaabi
  [Round of 16       ] Morocco              1–0 Tanzania            
    MOTM: Brahim Díaz
  [Quarter-finals    ] Cameroon             0–2 Morocco             
    MOTM: Ismael Saibari
  [Final             ] Senegal              0–3 Morocco             
    MOTM: Pape Gueye


In [14]:
# Show every row in Table 69 that mentions Morocco anywhere, regardless of stage label
flat_cols = [c[0] if isinstance(c, tuple) else c for c in tables_2025[69].columns]
t = tables_2025[69].copy()
t.columns = flat_cols

mask = (t['Team 1'].astype(str).str.contains('Morocco', na=False) |
        t['Team 2'].astype(str).str.contains('Morocco', na=False))
morocco_rows = t[mask]

print(f"All Table-69 rows mentioning Morocco ({len(morocco_rows)}):\n")
for idx, row in morocco_rows.iterrows():
    print(f"  Row {idx}: Stage='{row['Stage']}' | "
          f"{row['Team 1']} {row['Result']} {row['Team 2']} | "
          f"MOTM: {row['Man of the Match']}")

All Table-69 rows mentioning Morocco (7):

  Row 0: Stage='Group A' | Morocco 2–0 Comoros | MOTM: Brahim Díaz
  Row 15: Stage='Group A' | Morocco 1–1 Mali | MOTM: Neil El Aynaoui
  Row 27: Stage='Group A' | Zambia 0–3 Morocco | MOTM: Ayoub El Kaabi
  Row 39: Stage='Round of 16' | Morocco 1–0 Tanzania | MOTM: Brahim Díaz
  Row 46: Stage='Quarter-finals' | Cameroon 0–2 Morocco | MOTM: Ismael Saibari
  Row 50: Stage='Semi-finals' | Nigeria 0–0 (a.e.t.) (2–4 p) Morocco | MOTM: Yassine Bounou
  Row 52: Stage='Final' | Senegal 0–3 (w/o) Morocco | MOTM: Pape Gueye


In [15]:
def parse_score(score_str):
    """
    Parse a Wikipedia match score string into structured fields.
    Handles formats:
      "2-0"                       regular result
      "1-1 (a.e.t.)"              went to extra time, no penalties
      "0-0 (a.e.t.) (2-4 p)"     extra time + penalty shootout
      "1 (4)-1 (5)"               inline penalty shootout
      "0-3 (w/o)"                 walkover/awarded
    Returns dict, or None if unparseable.
    """
    if not score_str:
        return None
    s = str(score_str).strip()
   
    # Inline penalty format: "X (A)-Y (B)"
    m = re.match(
        r'^(\d+)\s*\((\d+)\)\s*[\-–]\s*(\d+)\s*\((\d+)\)(?:\s*\([^)]*\))*$',
        s,
    )
    if m:
        return {
            'home_score': int(m.group(1)), 'away_score': int(m.group(3)),
            'home_pen': int(m.group(2)),   'away_pen': int(m.group(4)),
            'penalty_shootout': True,
        }
   
    # Trailing-paren penalty format: "X-Y (a.e.t.) (A-B p)" or "X-Y (A-B p)"
    m = re.match(
        r'^(\d+)\s*[\-–]\s*(\d+)(?:\s*\([^)]*\))*\s*\((\d+)\s*[\-–]\s*(\d+)\s*p\)$',
        s,
    )
    if m:
        return {
            'home_score': int(m.group(1)), 'away_score': int(m.group(2)),
            'home_pen': int(m.group(3)),   'away_pen': int(m.group(4)),
            'penalty_shootout': True,
        }
   
    # Regular score with optional trailing annotations: "X-Y" or "X-Y (anything)"
    m = re.match(
        r'^(\d+)\s*[\-–]\s*(\d+)(?:\s*\([^)]*\))*$',
        s,
    )
    if m:
        return {
            'home_score': int(m.group(1)), 'away_score': int(m.group(2)),
            'home_pen': None, 'away_pen': None,
            'penalty_shootout': False,
        }
   
    return None


def parse_consolidated_match_table(t, focus_team='Morocco', tournament='AFCON', year=None):
    """Parse Wikipedia's row-per-match consolidated tables."""
    flat_cols = [c[0] if isinstance(c, tuple) else c for c in t.columns]
    t = t.copy()
    t.columns = flat_cols
   
    matches = []
    for _, row in t.iterrows():
        team1 = str(row.get('Team 1', '')).strip()
        team2 = str(row.get('Team 2', '')).strip()
        if focus_team not in (team1, team2):
            continue
       
        score_data = parse_score(row.get('Result', ''))
        if not score_data:
            continue
       
        # Strip "(p)" suffixes from team names
        team1 = re.sub(r'\s*\(p\)\s*$', '', team1).strip()
        team2 = re.sub(r'\s*\(p\)\s*$', '', team2).strip()
       
        matches.append({
            'tournament': tournament,
            'year': year,
            'stage': str(row.get('Stage', '')).strip(),
            'home_team': team1,
            'away_team': team2,
            **score_data,  # unpacks home_score, away_score, home_pen, away_pen, penalty_shootout
            'motm': str(row.get('Man of the Match', '')).strip(),
        })
    return matches


# Re-parse Morocco's AFCON 2025
morocco_2025 = parse_consolidated_match_table(
    tables_2025[69], focus_team='Morocco', tournament='AFCON', year=2025,
)

print(f"Morocco's AFCON 2025 — full path to the trophy ({len(morocco_2025)} matches):\n")
for m in morocco_2025:
    pen_note = f"  [P {m['home_pen']}-{m['away_pen']}]" if m['penalty_shootout'] else ""
    print(f"  [{m['stage']:18s}] {m['home_team']:20s} {m['home_score']}–{m['away_score']} {m['away_team']:20s}{pen_note}")
    if m['motm']:
        print(f"    MOTM: {m['motm']}")

Morocco's AFCON 2025 — full path to the trophy (7 matches):

  [Group A           ] Morocco              2–0 Comoros             
    MOTM: Brahim Díaz
  [Group A           ] Morocco              1–1 Mali                
    MOTM: Neil El Aynaoui
  [Group A           ] Zambia               0–3 Morocco             
    MOTM: Ayoub El Kaabi
  [Round of 16       ] Morocco              1–0 Tanzania            
    MOTM: Brahim Díaz
  [Quarter-finals    ] Cameroon             0–2 Morocco             
    MOTM: Ismael Saibari
  [Semi-finals       ] Nigeria              0–0 Morocco               [P 2-4]
    MOTM: Yassine Bounou
  [Final             ] Senegal              0–3 Morocco             
    MOTM: Pape Gueye


In [16]:
import time

# CAF qualifying for the 2026 World Cup
url = "https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_qualification_(CAF)"
print(f"Fetching: {url}\n")

try:
    qualifier_tables = fetch_wikipedia_tables(url)
    print(f"Returned {len(qualifier_tables)} tables\n")
except Exception as e:
    print(f"ERROR (will try alternate URL): {e}")
    # Try with en-dash URL pattern
    url = "https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_qualification_%E2%80%93_CAF"
    print(f"Retrying: {url}")
    qualifier_tables = fetch_wikipedia_tables(url)
    print(f"Returned {len(qualifier_tables)} tables\n")

# Try both parsers — Wikipedia might use the per-match (1, 3) format like AFCON 2023's
# group sub-articles, OR the consolidated row-per-match format like AFCON 2025's Table 69.
# We don't know yet, so we run both and combine.

# Approach 1: per-match (1, 3) tables
matches_v1 = parse_match_tables(qualifier_tables, stage_label='WC 2026 Q')
morocco_v1 = [m for m in matches_v1 if 'Morocco' in (m['home_team'], m['away_team'])]
print(f"Per-match parser found {len(matches_v1)} total matches, {len(morocco_v1)} involving Morocco")

# Approach 2: consolidated row-per-match table
morocco_v2 = []
for i, t in enumerate(qualifier_tables):
    if t.shape[1] >= 3 and t.shape[0] > 5:
        # Looks like it might be a consolidated table — try parser
        try:
            results = parse_consolidated_match_table(
                t, focus_team='Morocco',
                tournament='WC Qualifier', year=2026,
            )
            if results:
                print(f"  Consolidated parser found {len(results)} Morocco matches in Table {i}")
                morocco_v2.extend(results)
        except Exception:
            continue

print(f"\nCombined: {len(morocco_v1) + len(morocco_v2)} candidate Morocco matches")

# Combine and deduplicate by (home_team, away_team) pair
seen = set()
morocco_qualifiers = []
for m in morocco_v1 + morocco_v2:
    key = (m['home_team'], m['away_team'], m.get('home_score'), m.get('away_score'))
    if key not in seen:
        seen.add(key)
        morocco_qualifiers.append(m)

print(f"After dedup: {len(morocco_qualifiers)} unique Morocco qualifier matches\n")
for m in morocco_qualifiers:
    print(f"  {m['home_team']:20s} {m['home_score']}–{m['away_score']} {m['away_team']:20s}  ({m.get('stage', '')})")

Fetching: https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_qualification_(CAF)

Returned 18 tables

Per-match parser found 0 total matches, 0 involving Morocco

Combined: 0 candidate Morocco matches
After dedup: 0 unique Morocco qualifier matches



In [17]:
print(f"=== CAF WC 2026 qualifiers — all 18 tables ===\n")
for i, t in enumerate(qualifier_tables):
    cols = list(t.columns)
    cols_flat = [c[0] if isinstance(c, tuple) else c for c in cols]
    print(f"Table {i}: {t.shape[0]} rows × {t.shape[1]} cols")
    print(f"  Columns: {cols_flat[:8]}")
    if len(t) > 0:
        print(f"  First row: {t.iloc[0].to_list()[:8]}")
    print()

=== CAF WC 2026 qualifiers — all 18 tables ===

Table 0: 8 rows × 2 cols
  Columns: ['Tournament details', 'Tournament details.1']
  First row: ['Dates', '15 November 2023 – 16 November 2025']

Table 1: 3 rows × 1 cols
  Columns: ['Qualification for championships (CAF)']
  First row: ['FIFA World Cup 1958 1962 1966 1970 1974 1978 1982 1986 1990 1994 1998 2002 2006 2010 2014 2018 2022 2026 2030']

Table 2: 12 rows × 3 cols
  Columns: ['Round', 'Matchday', 'Dates']
  First row: ['First round', 'Matchday 1', '15–18 November 2023']

Table 3: 3 rows × 3 cols
  Columns: ['Pot 1', 'Pot 2', 'Pot 3']
  First row: ['Morocco (13)\xa0Senegal (18)\xa0Tunisia (31)\xa0Algeria (33)\xa0Egypt (34)\xa0Nigeria (39)\xa0Cameroon (43)\xa0Mali (50)\xa0Ivory Coast (51)', 'Burkina Faso (55)\xa0Ghana (59)\xa0South Africa (62)\xa0Cape Verde (66)\xa0DR Congo (69)\xa0Guinea (80)\xa0Zambia (84)\xa0Gabon (85)\xa0Equatorial Guinea (91)', 'Uganda (92)\xa0Benin (93)\xa0Mauritania (99)\xa0Kenya (105)\xa0Congo (106)\xa0Ma

In [18]:
import time

# Candidate URLs for the matches page — Wikipedia uses different conventions
# for tournament sub-pages. Try the most likely first.
candidate_urls = [
    "https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_qualification_(CAF)_second_round",
    "https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_qualification_(CAF)_%E2%80%93_Group_E",
    "https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_qualification_(CAF)_Group_E",
    "https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_qualification_%E2%80%93_CAF_second_round",
]

matches_page_tables = None
matches_page_url = None

for url in candidate_urls:
    print(f"Trying: {url}")
    try:
        tables = fetch_wikipedia_tables(url)
        # We need a page with actual MATCH data — should have many tables
        # and contain Morocco
        flat_text = ' '.join(
            ' '.join(str(c[0] if isinstance(c, tuple) else c) for c in t.columns) +
            ' '.join(str(v) for v in t.values.flatten() if not pd.isna(v))
            for t in tables
        )
        morocco_mentions = flat_text.count('Morocco')
        print(f"  ✓ {len(tables)} tables, {morocco_mentions} 'Morocco' mentions")
        if morocco_mentions > 3:  # threshold: real match data should mention Morocco many times
            matches_page_tables = tables
            matches_page_url = url
            break
    except Exception as e:
        print(f"  ✗ {e}")
    time.sleep(1)

if matches_page_tables:
    print(f"\n✓ Found it: {matches_page_url}")
    print(f"  {len(matches_page_tables)} tables to scan for matches\n")
   
    # Try both parsers on the matches page
    morocco_v1 = [m for m in parse_match_tables(matches_page_tables, stage_label='WC 2026 Q')
                  if 'Morocco' in (m['home_team'], m['away_team'])]
    morocco_v2 = []
    for t in matches_page_tables:
        if t.shape[1] >= 3 and t.shape[0] > 3:
            try:
                results = parse_consolidated_match_table(
                    t, focus_team='Morocco',
                    tournament='WC Qualifier', year=2026,
                )
                morocco_v2.extend(results)
            except Exception:
                continue
   
    # Dedup
    seen = set()
    morocco_qualifiers = []
    for m in morocco_v1 + morocco_v2:
        key = (m['home_team'], m['away_team'], m.get('home_score'), m.get('away_score'))
        if key not in seen:
            seen.add(key)
            morocco_qualifiers.append(m)
   
    print(f"Morocco's WC 2026 qualifying matches ({len(morocco_qualifiers)}):\n")
    for m in morocco_qualifiers:
        print(f"  {m['home_team']:20s} {m['home_score']}–{m['away_score']} {m['away_team']:20s}")
else:
    print(f"\n✗ None of the candidate URLs worked. We'll need to find the right one.")

Trying: https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_qualification_(CAF)_second_round
  ✗ 404 Client Error: Not Found for url: https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_qualification_(CAF)_second_round
Trying: https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_qualification_(CAF)_%E2%80%93_Group_E
  ✗ 404 Client Error: Not Found for url: https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_qualification_(CAF)_%E2%80%93_Group_E
Trying: https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_qualification_(CAF)_Group_E
  ✗ 404 Client Error: Not Found for url: https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_qualification_(CAF)_Group_E
Trying: https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_qualification_%E2%80%93_CAF_second_round
  ✓ 9 tables, 0 'Morocco' mentions

✗ None of the candidate URLs worked. We'll need to find the right one.


In [2]:
import re
import requests

# Define headers so Wikipedia doesn't block us like a bot
HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; soccer-data-project/1.0)"
}

# Fetch the parent CAF qualifiers page as raw HTML (not just tables)
parent_url = "https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_qualification_(CAF)"
print(f"Scanning HTML of: {parent_url}\n")
response = requests.get(parent_url, headers=HEADERS, timeout=30)
response.raise_for_status()
html = response.text

link_pattern = re.compile(r'href="(/wiki/[^"]+)"[^>]*>([^<]+)</a>')
links = link_pattern.findall(html)
print(f"Found {len(links)} total wiki links on the page. Filtering for relevant ones...\n")

keywords = [
    'Group_E', 'Group E',
    'second_round', 'second round',
    'first_round', 'first round',
    'third_round', 'third round',
    'matches', 'fixtures',
    'U-20', 'U20', 'under-20', 'under_20',
    'youth', 'CAF_U', 'caf_u'
]

seen = set()
relevant = []
for href, text in links:
    if href in seen:
        continue
    seen.add(href)
    haystack = (href + ' ' + text).lower()
    if any(kw.lower() in haystack for kw in keywords):
        relevant.append((href, text))

print(f"Found {len(relevant)} potentially-relevant links:\n")
for href, text in relevant[:25]:
    full_url = 'https://en.wikipedia.org' + href
    print(f"  {text[:45]:45s} → {full_url}")

Scanning HTML of: https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_qualification_(CAF)

Found 683 total wiki links on the page. Filtering for relevant ones...

Found 39 potentially-relevant links:

  Second round                                  → https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_qualification_%E2%80%93_CAF_second_round
  2026 FIFA World Cup qualification – CAF Group → https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_qualification_%E2%80%93_CAF_Group_E
  5–0                                           → https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_qualification_%E2%80%93_CAF_Group_E#MAR_v_NIG
  2–0                                           → https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_qualification_%E2%80%93_CAF_Group_E#MAR_v_TAN
  2–1                                           → https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_qualification_%E2%80%93_CAF_Group_E#MAR_v_ZAM
  1–0                                           → https://en.wikipedia.org/wiki/2026_F

In [4]:
import re
import time
import requests
import pandas as pd
from io import StringIO

# ── Setup ────────────────────────────────────────────────────────────────────

HEADERS = {
    'User-Agent': 'FootballIsLifeBot/0.1 (Personal soccer-analysis learning project; contact: cookyousfi@personal-email)'
}

SCORE_PATTERN = re.compile(r'^(\d+)\s*[\-–]\s*(\d+)(?:\s*\([^)]*\))*$')

def fetch_wikipedia_tables(url):
    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    return pd.read_html(StringIO(response.text))

def parse_score(score_str):
    if not score_str:
        return None
    s = str(score_str).strip()

    # Inline penalty: "X (A)-Y (B)"
    m = re.match(r'^(\d+)\s*\((\d+)\)\s*[\-–]\s*(\d+)\s*\((\d+)\)(?:\s*\([^)]*\))*$', s)
    if m:
        return {'home_score': int(m.group(1)), 'away_score': int(m.group(3)),
                'home_pen': int(m.group(2)), 'away_pen': int(m.group(4)),
                'penalty_shootout': True}

    # Trailing penalty: "X-Y (A-B p)"
    m = re.match(r'^(\d+)\s*[\-–]\s*(\d+)(?:\s*\([^)]*\))*\s*\((\d+)\s*[\-–]\s*(\d+)\s*p\)$', s)
    if m:
        return {'home_score': int(m.group(1)), 'away_score': int(m.group(2)),
                'home_pen': int(m.group(3)), 'away_pen': int(m.group(4)),
                'penalty_shootout': True}

    # Regular score
    m = re.match(r'^(\d+)\s*[\-–]\s*(\d+)(?:\s*\([^)]*\))*$', s)
    if m:
        return {'home_score': int(m.group(1)), 'away_score': int(m.group(2)),
                'home_pen': None, 'away_pen': None,
                'penalty_shootout': False}

    return None

def parse_match_tables(tables, stage_label=None):
    """Handles the (1,3) table format — score is the column header."""
    matches = []
    for t in tables:
        if t.shape != (1, 3):
            continue
        cols = [c[0] if isinstance(c, tuple) else c for c in t.columns]
        score_cell = str(cols[1]).strip()
        m = SCORE_PATTERN.match(score_cell)
        if not m:
            continue
        home_team = str(cols[0]).strip()
        away_team = str(cols[2]).strip()
        if home_team in ('', 'nan') or away_team in ('', 'nan'):
            continue
        matches.append({
            'stage': stage_label,
            'home_team': home_team,
            'away_team': away_team,
            'home_score': int(m.group(1)),
            'away_score': int(m.group(2)),
            'score_raw': score_cell,
        })
    return matches

def parse_consolidated_match_table(t, focus_team='Morocco', tournament='AFCON', year=None):
    """Handles row-per-match consolidated tables — score is a cell value."""
    flat_cols = [c[0] if isinstance(c, tuple) else c for c in t.columns]
    t = t.copy()
    t.columns = flat_cols
    matches = []
    for _, row in t.iterrows():
        team1 = str(row.get('Team 1', '')).strip()
        team2 = str(row.get('Team 2', '')).strip()
        if focus_team not in (team1, team2):
            continue
        score_data = parse_score(row.get('Result', ''))
        if not score_data:
            continue
        team1 = re.sub(r'\s*\(p\)\s*$', '', team1).strip()
        team2 = re.sub(r'\s*\(p\)\s*$', '', team2).strip()
        matches.append({
            'tournament': tournament,
            'year': year,
            'stage': str(row.get('Stage', '')).strip(),
            'home_team': team1,
            'away_team': team2,
            **score_data,
            'motm': str(row.get('Man of the Match', '')).strip(),
        })
    return matches

# ── Fetch Morocco WC 2026 Qualifier matches ──────────────────────────────────

qualifier_urls = {
    'Group E': 'https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_qualification_%E2%80%93_CAF_Group_E',
    'Second Round': 'https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_qualification_%E2%80%93_CAF_second_round',
}

morocco_wc_qualifiers = []

for stage_name, url in qualifier_urls.items():
    print(f"Fetching {stage_name}: {url}")
    try:
        tables = fetch_wikipedia_tables(url)
        print(f"  Got {len(tables)} tables")

        v1 = [m for m in parse_match_tables(tables, stage_label=stage_name)
              if 'Morocco' in (m['home_team'], m['away_team'])]

        v2 = []
        for t in tables:
            if t.shape[1] >= 3 and t.shape[0] > 3:
                try:
                    results = parse_consolidated_match_table(
                        t, focus_team='Morocco',
                        tournament='WC Qualifier', year=2026
                    )
                    v2.extend(results)
                except Exception:
                    continue

        seen = set()
        for m in v1 + v2:
            key = (m['home_team'], m['away_team'], m.get('home_score'), m.get('away_score'))
            if key not in seen:
                seen.add(key)
                m['tournament'] = 'WC Qualifier'
                m['year'] = 2026
                m['stage'] = stage_name
                morocco_wc_qualifiers.append(m)

        print(f"  Found {len(v1) + len(v2)} Morocco matches (before dedup)")
        time.sleep(1)

    except Exception as e:
        print(f"  ERROR: {e}")

print(f"\nMorocco WC 2026 Qualifiers ({len(morocco_wc_qualifiers)} matches):\n")
for m in morocco_wc_qualifiers:
    print(f"  [{m['stage']:12s}] {m['home_team']:20s} {m['home_score']}–{m['away_score']} {m['away_team']}")

Fetching Group E: https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_qualification_%E2%80%93_CAF_Group_E
  Got 34 tables
  Found 8 Morocco matches (before dedup)
Fetching Second Round: https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_qualification_%E2%80%93_CAF_second_round
  Got 9 tables
  Found 0 Morocco matches (before dedup)

Morocco WC 2026 Qualifiers (8 matches):

  [Group E     ] Tanzania             0–2 Morocco
  [Group E     ] Morocco              2–1 Zambia
  [Group E     ] Congo                0–6 Morocco
  [Group E     ] Niger                1–2 Morocco
  [Group E     ] Morocco              2–0 Tanzania
  [Group E     ] Morocco              5–0 Niger
  [Group E     ] Zambia               0–2 Morocco
  [Group E     ] Morocco              1–0 Congo


In [6]:
import re

# Test if our SCORE_PATTERN handles a.e.t. scores
SCORE_PATTERN = re.compile(r'^(\d+)\s*[\-–]\s*(\d+)(?:\s*\([^)]*\))*$')

test_scores = ['4–1 (a.e.t.)', '0–1', '1–1 (a.e.t.)', '4–1']
for s in test_scores:
    m = SCORE_PATTERN.match(s)
    print(f"  {s!r:25s} → {'MATCH ✓' if m else 'NO MATCH ✗'}")

  '4–1 (a.e.t.)'            → MATCH ✓
  '0–1'                     → MATCH ✓
  '1–1 (a.e.t.)'            → MATCH ✓
  '4–1'                     → MATCH ✓


In [11]:
morocco_wc_qualifiers = []

for stage_name, url in qualifier_urls.items():
    print(f"Fetching {stage_name}: {url}")
    try:
        tables = fetch_wikipedia_tables(url)
        print(f"  Got {len(tables)} tables")

        v1 = [m for m in parse_match_tables(tables, stage_label=stage_name)
              if 'Morocco' in (m['home_team'], m['away_team'])]

        v2 = []
        for t in tables:
            if t.shape[1] >= 3 and t.shape[0] > 3:
                try:
                    results = parse_consolidated_match_table(
                        t, focus_team='Morocco',
                        tournament='WC Qualifier', year=2026
                    )
                    v2.extend(results)
                except Exception:
                    continue

        seen = set()
        for m in v1 + v2:
            key = (m['home_team'], m['away_team'], m.get('home_score'), m.get('away_score'))
            if key not in seen:
                seen.add(key)
                m['tournament'] = 'WC Qualifier'
                m['year'] = 2026
                m['stage'] = stage_name
                morocco_wc_qualifiers.append(m)

        print(f"  Found {len(v1) + len(v2)} Morocco matches (before dedup)")
        time.sleep(1)

    except Exception as e:
        print(f"  ERROR: {e}")

print(f"\nMorocco WC 2026 Qualifiers ({len(morocco_wc_qualifiers)} matches):\n")
for m in morocco_wc_qualifiers:
    print(f"  [{m['stage']:12s}] {m['home_team']:20s} {m['home_score']}–{m['away_score']} {m['away_team']}")

Fetching Group E: https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_qualification_%E2%80%93_CAF_Group_E
  Got 34 tables
  Found 8 Morocco matches (before dedup)
Fetching Second Round: https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_qualification_%E2%80%93_CAF_second_round
  Got 9 tables
  Found 0 Morocco matches (before dedup)

Morocco WC 2026 Qualifiers (8 matches):

  [Group E     ] Tanzania             0–2 Morocco
  [Group E     ] Morocco              2–1 Zambia
  [Group E     ] Congo                0–6 Morocco
  [Group E     ] Niger                1–2 Morocco
  [Group E     ] Morocco              2–0 Tanzania
  [Group E     ] Morocco              5–0 Niger
  [Group E     ] Zambia               0–2 Morocco
  [Group E     ] Morocco              1–0 Congo


In [12]:
# ── Build master DataFrame ───────────────────────────────────────────────────

# AFCON 2023 — you already have df_2023 from earlier, let's just make sure
# it has the right columns
afcon_2023_rows = []
for m in afcon_2023_groups:
    afcon_2023_rows.append({
        'tournament': 'AFCON',
        'year': 2023,
        'stage': m['stage'],
        'home_team': m['home_team'],
        'away_team': m['away_team'],
        'home_score': m['home_score'],
        'away_score': m['away_score'],
        'home_pen': None,
        'away_pen': None,
        'penalty_shootout': False,
        'motm': '',
    })

# AFCON 2025 — you already have morocco_2025
afcon_2025_rows = []
for m in morocco_2025:
    afcon_2025_rows.append({
        'tournament': 'AFCON',
        'year': 2025,
        'stage': m['stage'],
        'home_team': m['home_team'],
        'away_team': m['away_team'],
        'home_score': m['home_score'],
        'away_score': m['away_score'],
        'home_pen': m.get('home_pen'),
        'away_pen': m.get('away_pen'),
        'penalty_shootout': m.get('penalty_shootout', False),
        'motm': m.get('motm', ''),
    })

# WC 2026 qualifiers
wc_qualifier_rows = []
for m in morocco_wc_qualifiers:
    wc_qualifier_rows.append({
        'tournament': 'WC Qualifier',
        'year': 2026,
        'stage': m['stage'],
        'home_team': m['home_team'],
        'away_team': m['away_team'],
        'home_score': m['home_score'],
        'away_score': m['away_score'],
        'home_pen': m.get('home_pen'),
        'away_pen': m.get('away_pen'),
        'penalty_shootout': m.get('penalty_shootout', False),
        'motm': m.get('motm', ''),
    })

# Combine everything
df_master = pd.DataFrame(afcon_2023_rows + afcon_2025_rows + wc_qualifier_rows)

# Add a helper column — did Morocco win?
def morocco_result(row):
    is_home = row['home_team'] == 'Morocco'
    is_away = row['away_team'] == 'Morocco'
    if not is_home and not is_away:
        return None  # Morocco not in this match
    mar_score = row['home_score'] if is_home else row['away_score']
    opp_score = row['away_score'] if is_home else row['home_score']
    if mar_score > opp_score:
        return 'W'
    elif mar_score < opp_score:
        return 'L'
    else:
        return 'D'

df_master['morocco_result'] = df_master.apply(morocco_result, axis=1)

print(f"Master DataFrame: {len(df_master)} total matches\n")
print(df_master.groupby(['tournament', 'year'])['morocco_result'].value_counts())
print(f"\nFull DataFrame:\n")
print(df_master.to_string(index=False))

NameError: name 'afcon_2023_groups' is not defined

In [15]:
import re
import time
import requests
import pandas as pd
from io import StringIO

# ── Constants ────────────────────────────────────────────────────────────────

HEADERS = {
    'User-Agent': 'FootballIsLifeBot/0.1 (Personal soccer-analysis learning project; contact: cookyousfi@personal-email)'
}

SCORE_PATTERN = re.compile(r'^(\d+)\s*[\-–]\s*(\d+)(?:\s*\([^)]*\))*$')

# ── Helper functions ─────────────────────────────────────────────────────────

def fetch_wikipedia_tables(url):
    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    return pd.read_html(StringIO(response.text))

def parse_score(score_str):
    if not score_str:
        return None
    s = str(score_str).strip()
    m = re.match(r'^(\d+)\s*\((\d+)\)\s*[\-–]\s*(\d+)\s*\((\d+)\)(?:\s*\([^)]*\))*$', s)
    if m:
        return {'home_score': int(m.group(1)), 'away_score': int(m.group(3)),
                'home_pen': int(m.group(2)), 'away_pen': int(m.group(4)),
                'penalty_shootout': True}
    m = re.match(r'^(\d+)\s*[\-–]\s*(\d+)(?:\s*\([^)]*\))*\s*\((\d+)\s*[\-–]\s*(\d+)\s*p\)$', s)
    if m:
        return {'home_score': int(m.group(1)), 'away_score': int(m.group(2)),
                'home_pen': int(m.group(3)), 'away_pen': int(m.group(4)),
                'penalty_shootout': True}
    m = re.match(r'^(\d+)\s*[\-–]\s*(\d+)(?:\s*\([^)]*\))*$', s)
    if m:
        return {'home_score': int(m.group(1)), 'away_score': int(m.group(2)),
                'home_pen': None, 'away_pen': None,
                'penalty_shootout': False}
    return None

def parse_match_tables(tables, stage_label=None, tournament=None, year=None):
    matches = []
    for t in tables:
        if t.shape != (1, 3):
            continue
        cols = [c[0] if isinstance(c, tuple) else c for c in t.columns]
        score_cell = str(cols[1]).strip()
        m = SCORE_PATTERN.match(score_cell)
        if not m:
            continue
        home_team = str(cols[0]).strip()
        away_team = str(cols[2]).strip()
        if home_team in ('', 'nan') or away_team in ('', 'nan'):
            continue
        matches.append({
            'tournament': tournament,
            'year': year,
            'stage': stage_label,
            'home_team': home_team,
            'away_team': away_team,
            'home_score': int(m.group(1)),
            'away_score': int(m.group(2)),
            'home_pen': None,
            'away_pen': None,
            'penalty_shootout': False,
            'motm': '',
        })
    return matches

def parse_consolidated_match_table(t, focus_team='Morocco', tournament=None, year=None):
    flat_cols = [c[0] if isinstance(c, tuple) else c for c in t.columns]
    t = t.copy()
    t.columns = flat_cols
    score_col = next((c for c in ('Result', 'Score', 'score', 'result') if c in flat_cols), None)
    if score_col is None:
        return []
    matches = []
    for _, row in t.iterrows():
        team1 = str(row.get('Team 1', '')).strip()
        team2 = str(row.get('Team 2', '')).strip()
        if focus_team not in (team1, team2):
            continue
        score_data = parse_score(row.get(score_col, ''))
        if not score_data:
            continue
        team1 = re.sub(r'\s*\(p\)\s*$', '', team1).strip()
        team2 = re.sub(r'\s*\(p\)\s*$', '', team2).strip()
        matches.append({
            'tournament': tournament,
            'year': year,
            'stage': str(row.get('Stage', '')).strip(),
            'home_team': team1,
            'away_team': team2,
            **score_data,
            'motm': str(row.get('Man of the Match', '')).strip(),
        })
    return matches

def fetch_matches_from_url(url, stage_name, focus_team='Morocco', tournament=None, year=None):
    """Fetch a Wikipedia page and extract all matches involving focus_team."""
    print(f"  Fetching {stage_name}...", end=' ', flush=True)
    try:
        tables = fetch_wikipedia_tables(url)
    except Exception as e:
        print(f"FAILED: {e}")
        return []

    v1 = [m for m in parse_match_tables(tables, stage_label=stage_name, tournament=tournament, year=year)
          if focus_team in (m['home_team'], m['away_team'])]

    v2 = []
    for t in tables:
        if t.shape[1] >= 3 and t.shape[0] > 3:
            try:
                v2.extend(parse_consolidated_match_table(
                    t, focus_team=focus_team, tournament=tournament, year=year
                ))
            except Exception:
                continue

    seen = set()
    results = []
    for m in v1 + v2:
        key = (m['home_team'], m['away_team'], m.get('home_score'), m.get('away_score'))
        if key not in seen:
            seen.add(key)
            m['stage'] = stage_name
            results.append(m)

    print(f"{len(results)} matches")
    time.sleep(1)
    return results

def morocco_result(row):
    is_home = row['home_team'] == 'Morocco'
    is_away = row['away_team'] == 'Morocco'
    if not is_home and not is_away:
        return None
    mar = row['home_score'] if is_home else row['away_score']
    opp = row['away_score'] if is_home else row['home_score']
    return 'W' if mar > opp else ('L' if mar < opp else 'D')

# ── Fetch all data ───────────────────────────────────────────────────────────

all_matches = []

print("=== AFCON 2023 ===")
# Group stage
for letter in 'ABCDEF':
    all_matches.extend(fetch_matches_from_url(
        url=f"https://en.wikipedia.org/wiki/2023_Africa_Cup_of_Nations_Group_{letter}",
        stage_name=f'Group {letter}', focus_team='Morocco',
        tournament='AFCON', year=2023
    ))

# Knockout stage
all_matches.extend(fetch_matches_from_url(
    url="https://en.wikipedia.org/wiki/2023_Africa_Cup_of_Nations_knockout_stage",
    stage_name='Round of 16', focus_team='Morocco',
    tournament='AFCON', year=2023
))

print("\n=== AFCON 2025 ===")
tables_2025 = fetch_wikipedia_tables("https://en.wikipedia.org/wiki/2025_Africa_Cup_of_Nations")
morocco_2025 = parse_consolidated_match_table(
    tables_2025[69], focus_team='Morocco', tournament='AFCON', year=2025
)
all_matches.extend(morocco_2025)
print(f"  AFCON 2025: {len(morocco_2025)} Morocco matches")

print("\n=== WC 2026 Qualifiers ===")
for stage_name, url in {
    'Group E': 'https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_qualification_%E2%80%93_CAF_Group_E',
    'Second Round': 'https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_qualification_%E2%80%93_CAF_second_round',
}.items():
    all_matches.extend(fetch_matches_from_url(
        url=url, stage_name=stage_name, focus_team='Morocco',
        tournament='WC Qualifier', year=2026
    ))

# ── Build master DataFrame ───────────────────────────────────────────────────

df_master = pd.DataFrame(all_matches)
df_master['morocco_result'] = df_master.apply(morocco_result, axis=1)

print(f"\n{'='*50}")
print(f"Master DataFrame: {len(df_master)} total matches\n")
print(df_master.groupby(['tournament', 'year'])['morocco_result'].value_counts())
print(f"\n{df_master.to_string(index=False)}")

=== AFCON 2023 ===
  Fetching Group A... 0 matches
  Fetching Group B... 0 matches
  Fetching Group C... 0 matches
  Fetching Group D... 0 matches
  Fetching Group E... 0 matches
  Fetching Group F... 3 matches
  Fetching Round of 16... 1 matches

=== AFCON 2025 ===
  AFCON 2025: 7 Morocco matches

=== WC 2026 Qualifiers ===
  Fetching Group E... 8 matches
  Fetching Second Round... 0 matches

Master DataFrame: 19 total matches

tournament    year  morocco_result
AFCON         2023  W                 2
                    D                 1
                    L                 1
              2025  W                 5
                    D                 2
WC Qualifier  2026  W                 8
Name: count, dtype: int64

  tournament  year          stage home_team    away_team  home_score  away_score  home_pen  away_pen  penalty_shootout            motm morocco_result
       AFCON  2023        Group F   Morocco     Tanzania           3           0       NaN       NaN             Fa